In [3]:
import pandas as pd
from pathlib import Path

FILE_PATH = Path(
    r"C:\Users\KIIT\OneDrive\Desktop\Financial Analytics Project\Raw Data\financial_ratios.xlsx"
)

raw = pd.read_excel(
    FILE_PATH,
    sheet_name="Sheet1"
)

print("Shape:", raw.shape)

print("\nColumns:")
print(raw.columns.tolist())

print("\nData types:")
print(raw.dtypes)

print("\nMissing values:")
print(raw.isna().sum())

print("\nDuplicate IDs:")
print(raw["id"].duplicated().sum())

print("\nDuplicate company-year:")
print(
    raw.duplicated(
        subset=["company_id", "year"]
    ).sum()
)

print("\nYear range:")
print(raw["year"].min(), "to", raw["year"].max())

Shape: (1184, 16)

Columns:
['id', 'company_id', 'year', 'net_profit_margin_pct', 'operating_profit_margin_pct', 'return_on_equity_pct', 'debt_to_equity', 'interest_coverage', 'asset_turnover', 'free_cash_flow_cr', 'capex_cr', 'earnings_per_share', 'book_value_per_share', 'dividend_payout_ratio_pct', 'total_debt_cr', 'cash_from_operations_cr']

Data types:
id                               int64
company_id                      object
year                            object
net_profit_margin_pct          float64
operating_profit_margin_pct    float64
return_on_equity_pct           float64
debt_to_equity                 float64
interest_coverage              float64
asset_turnover                 float64
free_cash_flow_cr              float64
capex_cr                       float64
earnings_per_share             float64
book_value_per_share           float64
dividend_payout_ratio_pct      float64
total_debt_cr                    int64
cash_from_operations_cr        float64
dtype: object

Mi

In [4]:
raw.describe().T

,count,mean,std,min,25%,50%,75%,max
id,1184.0,592.500000,341.935666,1.0000,296.750000,592.50000,888.250000,1.184000e+03
net_profit_margin_pct,1183.0,18.656940,67.670474,-42.1500,5.225000,11.04000,17.070000,9.475100e+02
operating_profit_margin_pct,1123.0,346.891362,5504.872522,-50971.0000,13.000000,21.00000,39.000000,4.797100e+04
return_on_equity_pct,1184.0,152.763970,1145.252160,-1094.1500,10.555000,16.71000,24.950000,1.788636e+04
debt_to_equity,1184.0,2.811181,12.703534,0.0000,0.050650,0.57695,2.573025,3.325938e+02
interest_coverage,1091.0,133.613752,630.262608,-4754.5000,2.915750,11.66670,56.346500,6.250000e+03
asset_turnover,1183.0,4.110240,21.980365,0.0004,0.235300,0.69210,1.171700,3.539615e+02
free_cash_flow_cr,1182.0,779.983080,16173.746988,-115427.0000,-713.000000,741.00000,3640.500000,2.290490e+05
capex_cr,1182.0,6056.598139,13460.345951,0.0000,503.500000,1526.50000,5464.250000,1.484470e+05
earnings_per_share,1180.0,115.895763,1512.606350,-160.0000,7.000000,18.00000,41.000000,4.248100e+04


In [5]:
# Missing values
missing = pd.DataFrame({
    "missing_count": raw.isna().sum(),
    "missing_pct": raw.isna().mean() * 100
}).sort_values("missing_count", ascending=False)

print(missing)

                             missing_count  missing_pct
interest_coverage                       93     7.854730
operating_profit_margin_pct             61     5.152027
earnings_per_share                       4     0.337838
dividend_payout_ratio_pct                4     0.337838
free_cash_flow_cr                        2     0.168919
capex_cr                                 2     0.168919
cash_from_operations_cr                  2     0.168919
net_profit_margin_pct                    1     0.084459
asset_turnover                           1     0.084459
id                                       0     0.000000
company_id                               0     0.000000
year                                     0     0.000000
return_on_equity_pct                     0     0.000000
debt_to_equity                           0     0.000000
book_value_per_share                     0     0.000000
total_debt_cr                            0     0.000000


In [6]:
# Duplicate checks
print("Duplicate IDs:", raw["id"].duplicated().sum())

print(
    "Duplicate company-year:",
    raw.duplicated(
        subset=["company_id", "year"]
    ).sum()
)

duplicate_company_year = raw[
    raw.duplicated(
        subset=["company_id", "year"],
        keep=False
    )
].sort_values(["company_id", "year"])

print(duplicate_company_year)

Duplicate IDs: 0
Duplicate company-year: 119
        id company_id      year  net_profit_margin_pct  \
1        2        ABB  Mar 2014                   8.70   
2        3        ABB  Mar 2014                   8.70   
3        4        ABB  Mar 2015                  10.00   
4        5        ABB  Mar 2015                  10.00   
5        6        ABB  Mar 2016                   9.76   
...    ...        ...       ...                    ...   
1095  1096      TECHM  Mar 2022                  12.61   
1096  1097      TECHM  Mar 2023                   9.11   
1097  1098      TECHM  Mar 2023                   9.11   
1098  1099      TECHM  Mar 2024                   4.61   
1099  1100      TECHM  Mar 2024                   4.61   

      operating_profit_margin_pct  return_on_equity_pct  debt_to_equity  \
1                            12.0                 25.13          0.0000   
2                            12.0                 25.13          0.0000   
3                            14.0

In [7]:
checks = {
    "Debt < 0": (raw["total_debt_cr"] < 0).sum(),
    "Capex < 0": (raw["capex_cr"] < 0).sum(),
    "Book value < 0": (raw["book_value_per_share"] < 0).sum(),
    "D/E < 0": (raw["debt_to_equity"] < 0).sum(),
    "Asset turnover < 0": (raw["asset_turnover"] < 0).sum(),
}

for check, count in checks.items():
    print(f"{check}: {count}")

Debt < 0: 0
Capex < 0: 0
Book value < 0: 0
D/E < 0: 0
Asset turnover < 0: 0


In [8]:
percentage_checks = {
    "Net margin < -100%": (raw["net_profit_margin_pct"] < -100).sum(),
    "Operating margin < -100%": (
        raw["operating_profit_margin_pct"] < -100
    ).sum(),
    "ROE < -100%": (
        raw["return_on_equity_pct"] < -100
    ).sum(),
    "Dividend payout < -100%": (
        raw["dividend_payout_ratio_pct"] < -100
    ).sum(),
}

for check, count in percentage_checks.items():
    print(f"{check}: {count}")

Net margin < -100%: 0
Operating margin < -100%: 62
ROE < -100%: 19
Dividend payout < -100%: 9


In [9]:
dup = raw[
    raw.duplicated(
        subset=["company_id", "year"],
        keep=False
    )
].sort_values(["company_id", "year"])

# Check whether each company-year has only one unique financial record
financial_cols = [
    c for c in raw.columns
    if c not in ["id", "company_id", "year"]
]

conflicts = (
    dup.groupby(["company_id", "year"])[financial_cols]
       .nunique(dropna=False)
       .gt(1)
       .any(axis=1)
)

print("Duplicate company-year groups:", len(conflicts))
print("Exact duplicate groups:", (~conflicts).sum())
print("Conflicting groups:", conflicts.sum())

Duplicate company-year groups: 83
Exact duplicate groups: 72
Conflicting groups: 11


In [10]:
kpi = raw.drop_duplicates(
    subset=["company_id", "year"],
    keep="first"
).reset_index(drop=True)

print("Rows before:", len(raw))
print("Rows after:", len(kpi))
print(
    "Duplicate company-year after cleaning:",
    kpi.duplicated(["company_id", "year"]).sum()
)

Rows before: 1184
Rows after: 1065
Duplicate company-year after cleaning: 0


In [11]:
extreme = kpi[
    (kpi["operating_profit_margin_pct"] < -100) |
    (kpi["return_on_equity_pct"] < -100) |
    (kpi["dividend_payout_ratio_pct"] < -100)
]

print("Extreme financial observations:", len(extreme))

print(
    extreme[
        [
            "company_id",
            "year",
            "operating_profit_margin_pct",
            "return_on_equity_pct",
            "dividend_payout_ratio_pct"
        ]
    ].to_string(index=False)
)

Extreme financial observations: 78
company_id     year  operating_profit_margin_pct  return_on_equity_pct  dividend_payout_ratio_pct
ADANIPOWER Mar 2017                         26.0               -205.80                        0.0
ADANIPOWER Mar 2018                         27.0               -236.56                        0.0
  AXISBANK Mar 2017                      -5715.0                  6.53                       33.0
  AXISBANK Mar 2018                     -10277.0                  0.43                        0.0
  AXISBANK Mar 2019                      -5447.0                  6.90                        6.0
  AXISBANK Mar 2020                      -9859.0                  1.88                        0.0
  AXISBANK Mar 2021                      -2510.0                  6.36                        0.0
BANKBARODA Mar 2016                     -11700.0                -11.84                        0.0
BANKBARODA Mar 2017                      -4372.0                  4.31             

In [12]:
conflicting_groups = conflicts[conflicts].reset_index()

print("Conflicting company-year groups:")
print(conflicting_groups)

Conflicting company-year groups:
   company_id      year     0
0         ABB  Mar 2014  True
1         ABB  Mar 2015  True
2         ABB  Mar 2016  True
3         ABB  Mar 2017  True
4         ABB  Mar 2018  True
5         ABB  Mar 2019  True
6         ABB  Mar 2020  True
7         ABB  Mar 2021  True
8         ABB  Mar 2022  True
9         ABB  Mar 2023  True
10        ABB  Mar 2024  True


In [13]:
conflicting_records = raw.merge(
    conflicting_groups[["company_id", "year"]],
    on=["company_id", "year"],
    how="inner"
).sort_values(["company_id", "year"])

print(
    conflicting_records.to_string(index=False)
)

 id company_id     year  net_profit_margin_pct  operating_profit_margin_pct  return_on_equity_pct  debt_to_equity  interest_coverage  asset_turnover  free_cash_flow_cr  capex_cr  earnings_per_share  book_value_per_share  dividend_payout_ratio_pct  total_debt_cr  cash_from_operations_cr
  2        ABB Mar 2014                   8.70                         12.0                 25.13          0.0000                NaN          1.9982               11.0     144.0                93.0                3.7524                       25.0              0                    155.0
  3        ABB Mar 2014                   8.70                         12.0                 25.13          0.0000                NaN          1.9982                0.0       0.0                93.0                3.7524                       25.0              0                      0.0
  4        ABB Mar 2015                  10.00                         14.0                 24.44          0.0000                NaN       

In [14]:
# Compare the conflicting records while focusing on the fields
# that actually differ.

for (company, year), group in conflicting_records.groupby(
    ["company_id", "year"]
):
    print(f"\n{'='*60}")
    print(company, year)

    for col in financial_cols:
        if group[col].nunique(dropna=False) > 1:
            print(
                f"{col}:",
                group[col].tolist()
            )


ABB Mar 2014
free_cash_flow_cr: [11.0, 0.0]
capex_cr: [144.0, 0.0]
cash_from_operations_cr: [155.0, 0.0]

ABB Mar 2015
free_cash_flow_cr: [28.0, -1899.0]
capex_cr: [187.0, 1864.0]
cash_from_operations_cr: [215.0, -35.0]

ABB Mar 2016
free_cash_flow_cr: [172.0, 728.0]
capex_cr: [77.0, 816.0]
cash_from_operations_cr: [249.0, 1544.0]

ABB Mar 2017
free_cash_flow_cr: [152.0, 459.0]
capex_cr: [155.0, 1730.0]
cash_from_operations_cr: [307.0, 2189.0]

ABB Mar 2018
free_cash_flow_cr: [-62.0, -994.0]
capex_cr: [215.0, 3192.0]
cash_from_operations_cr: [153.0, 2198.0]

ABB Mar 2019
free_cash_flow_cr: [242.0, -459.0]
capex_cr: [257.0, 3050.0]
cash_from_operations_cr: [499.0, 2591.0]

ABB Mar 2020
free_cash_flow_cr: [225.0, -206.0]
capex_cr: [401.0, 5643.0]
cash_from_operations_cr: [626.0, 5437.0]

ABB Mar 2021
free_cash_flow_cr: [655.0, -225.0]
capex_cr: [72.0, 4009.0]
cash_from_operations_cr: [727.0, 3784.0]

ABB Mar 2022
free_cash_flow_cr: [552.0, 161.0]
capex_cr: [396.0, 3936.0]
cash_from_oper

In [15]:
kpi_clean = raw.drop_duplicates(
    subset=["company_id", "year"],
    keep="first"
).reset_index(drop=True)

print("Rows before:", len(raw))
print("Rows after:", len(kpi_clean))
print(
    "Duplicate company-year after cleaning:",
    kpi_clean.duplicated(["company_id", "year"]).sum()
)

Rows before: 1184
Rows after: 1065
Duplicate company-year after cleaning: 0


In [16]:
from pathlib import Path

KPI_CLEANED = Path(
    r"C:\Users\KIIT\OneDrive\Desktop\Financial Analytics Project\Cleaned Data"
)

KPI_CLEANED.mkdir(parents=True, exist_ok=True)

kpi_clean.to_csv(
    KPI_CLEANED / "financial_kpis_clean.csv",
    index=False
)

print("Saved:", KPI_CLEANED / "financial_kpis_clean.csv")

Saved: C:\Users\KIIT\OneDrive\Desktop\Financial Analytics Project\Cleaned Data\financial_kpis_clean.csv


In [17]:
import pandas as pd

ratios = pd.read_csv(
    r"C:\Users\KIIT\OneDrive\Desktop\Financial Analytics Project\Cleaned Data\financial_kpis_clean.csv"
)

print("Shape:", ratios.shape)
print("Columns:", ratios.columns.tolist())

print("\nMissing values:")
print(ratios.isna().sum())

print("\nDuplicate IDs:", ratios["id"].duplicated().sum())

print("\nUnique companies:", ratios["company_id"].nunique())

Shape: (1065, 16)
Columns: ['id', 'company_id', 'year', 'net_profit_margin_pct', 'operating_profit_margin_pct', 'return_on_equity_pct', 'debt_to_equity', 'interest_coverage', 'asset_turnover', 'free_cash_flow_cr', 'capex_cr', 'earnings_per_share', 'book_value_per_share', 'dividend_payout_ratio_pct', 'total_debt_cr', 'cash_from_operations_cr']

Missing values:
id                              0
company_id                      0
year                            0
net_profit_margin_pct           1
operating_profit_margin_pct    13
return_on_equity_pct            0
debt_to_equity                  0
interest_coverage              43
asset_turnover                  1
free_cash_flow_cr               2
capex_cr                        2
earnings_per_share              4
book_value_per_share            0
dividend_payout_ratio_pct       4
total_debt_cr                   0
cash_from_operations_cr         2
dtype: int64

Duplicate IDs: 0

Unique companies: 92
